In [ ]:
import numpy as np
from pyscf import gto,scf,mcscf
from pyscf.data.nist import HARTREE2EV as au2ev

def get_h2(r, basis='cc-pvdz'):

    h2 = gto.M()
    h2.atom = [['H', (0.,0.,0.)],['H', (0.,0.,r)]]
    h2.basis = basis
    h2.verbose = 4
    h2.symmetry = 1
    h2.build()
    
    return h2

In [ ]:
r0 = .74144; rr = 2

r = r0*rr

h2 = get_h2(r, basis='cc-pvtz')

rhf = scf.RHF(h2)
rhf.verbose = 6
rhf.kernel()

print('RHF mo_energy, ', rhf.mo_energy)
print('RHF IP =%8.3f'%(-rhf.mo_energy[0]*au2ev))
print('RHF EA =%8.3f'%(rhf.mo_energy[1]*au2ev))

rhf.analyze()

In [ ]:
from pyscf import fci

from gf.addons import costru_h1e_h2e, exdiag_nev, exdiag_nev_diagss
from gf.gw.addons import gf_from_FCI

nmo = rhf.mo_coeff.shape[1]
nelec = (1,1)

eta = .1/au2ev

# ==== unavailable on this computer ====

#h1e, h2e = costru_h1e_h2e(h2, rhf.mo_coeff, rhf._eri)
#Gfci = gf_from_FCI(h1e, h2e, nmo, nelec, eta=eta)

#cisolver = fci.addons.fix_spin( fci.direct_spin1.FCI() )



In [ ]:
from gf.gw.srgw import selfen_GW_RHF, gf_from_selfen

s = selfen_GW_RHF(rhf, eta=eta)
Gsrgw = gf_from_selfen(rhf, s, eta=eta)


In [ ]:
from gf.gw.mrgw import CASGW

cas_dict = {'a1g':1, 'a1u':1}

casscf = mcscf.CASSCF(rhf, 2, (1,1))
#casscf = mcscf.CASCI(rhf, 2, (1,1))
casscf.fix_spin_(ss=0.)
mo_coeff = mcscf.sort_mo_by_irrep(casscf, rhf.mo_coeff, cas_dict)
casscf.kernel(mo_coeff)

print('   det-alpha,    det-beta,    CI coefficients')
for c,ia,ib in casscf.fcisolver.large_ci(casscf.ci, 2, (1,1), tol=0.05, return_strs=False):
    print('   %s       %s      %.12f' % (ia, ib, c))

mrgw = CASGW(casscf, ss=0.)
Gcas, Gmrgw = mrgw.G(eta=eta)


In [ ]:
from gf.gw.addons import spec_from_gf

spec_srgw = spec_from_gf(Gsrgw)
#spec_fci = spec_from_gf(Gfci)
spec_cas = spec_from_gf(Gcas)
spec_mrgw = spec_from_gf(Gmrgw)


In [ ]:
from gf.gw.addons import wrt_gf_dat

wrt_gf_dat(Gsrgw, -2, 1, .2*eta, f'spec-GW@RHF-{rr:2.1f}.dat')
wrt_gf_dat(Gcas, -2, 1, .2*eta, f'spec-CAS-{rr:2.1f}.dat')
wrt_gf_dat(Gmrgw, -2, 1, .2*eta, f'spec-GW@CAS-{rr:2.1f}.dat')
#wrt_gf_dat(Gfci, -2, 1, .2*eta, f'spec-FCI-{rr:2.1f}.dat')

In [ ]:
#from gf.gw.addons import wrt_gf_dat
#wrt_gf_dat(Gmrsosex, -2, 1, .2*eta, f'spec-SOSEX@CAS-{rr:2.1f}.dat')

In [ ]:
W = np.arange(-1, 1, .2*eta)
dat_srgw = [spec_srgw(x) for x in W]
dat_cas = [spec_cas(x) for x in W]
#dat_fci = [spec_fci(x) for x in W]
dat_mrgw = [spec_mrgw(x) for x in W]


In [ ]:
from pyscf.lib import logger


def stable_opt_internal(mf):
    log = logger.new_logger(mf)
    mo1, _, stable, _ = mf.stability(return_status=True)
    cyc = 0
    while (not stable and cyc < 10):
        log.note('Try to optimize orbitals until stable, attempt %d' % cyc)
        dm1 = mf.make_rdm1(mo1, mf.mo_occ)
        mf = mf.run(dm1)
        mo1, _, stable, _ = mf.stability(return_status=True, tol=1e-7)
        cyc += 1
    if not stable:
        log.note('Stability Opt failed after %d attempts' % cyc)
    return mf

In [ ]:
from gf.gw.srgw import gf_gw_from_uhf

r0 = .74144; rr = 2

r = r0*rr

h2 = get_h2(r, basis='cc-pvtz')

uhf = scf.uhf.UHF(h2)
uhf.verbose = 6
uhf.kernel()

mo_a, mo_b = uhf.mo_coeff
mo_b[:,0], mo_b[:,1] = mo_b[:,1], mo_b[:,0]

dma = np.dot(mo_a[:,:1], mo_a[:,:1].T)
dmb = np.dot(mo_b[:,:1], mo_b[:,:1].T)

uhf.kernel(dm=(dma,dmb))

uhf = stable_opt_internal(uhf)

uhf.stability()

uhf.analyze()

print(uhf.mo_energy)


eta = .1/au2ev

Gsrgw = gf_gw_from_uhf(uhf, eta=eta)
spec_srgw = spec_from_gf(Gsrgw)

In [ ]:
from gf.gw.addons import wrt_gf_dat

wrt_gf_dat(Gsrgw, -1, 1, .2*eta, f'spec-GW@UHF-{rr:2.1f}.dat')